# Token Model — Pretrained CodeGen-350M-mono

In [1]:
%tb
import os, json, math, random, glob
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from tqdm import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

from modules.plotting import MetricLog, plot_metrics
from modules.early_stopping import EarlyStopping
from modules.hand_testing import hand_test_repl
from modules.best_model_saver import BestModelSaver
from modules.datasets.loading import *
from modules.datasets.token_codegen_dataset import *


import warnings
warnings.filterwarnings("ignore")

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")

HF_MODEL_NAME    = "Salesforce/codegen-350M-mono"
TOKEN_MODEL_NAME = 'token_model_codegen'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")

No traceback available to show.


WORKDIR: C:\Users\Roman\Documents\Projects\code_autocomplete
[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA: 12.4
Количество GPU: 1


## Training loop

* Mixed precision via `autocast("cuda", dtype=torch.float16)` + `GradScaler`
* Warmup + cosine schedule via `get_cosine_schedule_with_warmup` (steps per batch)
* Label smoothing in the cross-entropy loss (`label_smoothing=0.1`)
* Early stopping on validation loss

In [2]:
def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:           nn.Module,
    hf_tokenizer,              
    train_dl:        DataLoader,
    val_dl:          DataLoader,
    epochs:          int,
    lr:              float,
    device:          torch.device,
    saver:           BestModelSaver,
    log:             MetricLog,
    plot_dir:        str,
    label_smoothing: float = 0.1,
    warmup_frac:     float = 0.05,
    patience:        int   = 3,
    use_amp:         bool  = True,
):
    tqdm.write(f"[Token] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")

    # ── optimiser + warmup-cosine schedule ───────────────────────────────────
    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    # ── mixed precision scaler (no-op on CPU) ────────────────────────────────
    amp_enabled = use_amp and device.type == "cuda"
    scaler      = GradScaler("cuda", enabled=amp_enabled)

    # ── early stopping ───────────────────────────────────────────────────────
    stopper   = EarlyStopping(patience=patience)
    pad_id    = hf_tokenizer.pad_token_id if hf_tokenizer.pad_token_id is not None else -100
    epoch_bar = tqdm(range(1, epochs + 1), desc="[Token] Epochs", unit="ep", position=0)

    for ep in epoch_bar:
        # ── TRAIN ────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_acc = t_steps = 0
        gn = 0.0

        batch_bar = tqdm(train_dl,
                         desc=f"[Token] Epoch {ep}/{epochs} train",
                         leave=False, unit="batch", position=1)

        for chunk in batch_bar:
            # chunk shape: (B, ctx+1).  Build (input, target) by shifting.
            chunk = chunk.to(device, non_blocking=True)
            x = chunk[:, :-1]              # (B, ctx)
            y = chunk[:,  1:]              # (B, ctx) — next-token targets

            opt.zero_grad(set_to_none=True)

            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(input_ids=x).logits          # (B, ctx, V)
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    y.reshape(-1),
                    label_smoothing=label_smoothing,
                    ignore_index=pad_id if pad_id != -100 else -100,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt)
            scaler.update()
            sched.step()                  # per-batch step for warmup+cosine

            with torch.no_grad():
                preds = logits.argmax(-1)
                valid = (y != pad_id) if pad_id != -100 else torch.ones_like(y, dtype=torch.bool)
                acc   = (preds[valid] == y[valid]).float().mean().item() if valid.any() else 0.0

            t_loss  += loss.item()
            t_acc   += acc
            t_steps += 1

            batch_bar.set_postfix(loss=f"{loss.item():.4f}",
                                  acc=f"{acc:.3f}",
                                  lr=f"{opt.param_groups[0]['lr']:.2e}")

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        # ── VAL ──────────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for chunk in tqdm(val_dl,
                              desc=f"[Token] Epoch {ep}/{epochs} val  ",
                              leave=False, unit="batch", position=1):
                chunk = chunk.to(device, non_blocking=True)
                x, y  = chunk[:, :-1], chunk[:, 1:]
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    logits = model(input_ids=x).logits
                    loss   = F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        y.reshape(-1),
                        label_smoothing=label_smoothing,
                        ignore_index=pad_id if pad_id != -100 else -100,
                    )
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl

        # ── LOGGING ──────────────────────────────────────────────────────────
        log.append(
            train_loss=tl, val_loss=vl,
            train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
            lr=opt.param_groups[0]["lr"],
            token_acc=ta, grad_norm=gn,
        )
        epoch_bar.set_postfix(
            train=f"{tl:.4f}", val=f"{vl:.4f}",
            ppl=f"{math.exp(min(vl, 20)):.1f}",
        )
        tqdm.write(
            f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
            f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
            f"  lr={opt.param_groups[0]['lr']:.2e}"
        )

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(
                log,
                f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                f"{plot_dir}/{TOKEN_MODEL_NAME}_ep{ep:02d}.png",
            )

        # ── EARLY STOPPING ───────────────────────────────────────────────────
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break

    plot_metrics(
        log,
        f"{TOKEN_MODEL_NAME.replace('_', ' ')} — Final",
        f"{plot_dir}/{TOKEN_MODEL_NAME}_final.png",
    )

## Inline hand-test for CodeGen

Your existing `hand_test_repl` was written for the custom `TokenModel.generate(prefix_ids, ...)` API. CodeGen uses HuggingFace's `model.generate(input_ids=...)` instead, so this small REPL replaces it for testing without touching `modules.HandTesting`.

In [3]:
# def hand_test_codegen(model, hf_tok, device: torch.device):
#     """
#     Commands:
#       :token  <prefix>  → next-word completion (stops at first whitespace)
#       :line   <prefix>  → multi-token completion (until newline)
#       :temp   <float>   → set sampling temperature
#       :k      <int>     → set top-k
#       :quit             → exit
#     """
#     print("  CodeGen Token Model — Interactive Test")
#     print("  Commands: :token <prefix>  |  :line <prefix>  |  :temp <f>  |  :k <i>  |  :quit")
#     temperature = 0.3
#     top_k       = 10
#     model.eval()

#     while True:
#         try:
#             raw = input(">> ").strip()
#         except (EOFError, KeyboardInterrupt):
#             print("\nBye!"); break
#         if not raw:
#             continue
#         if raw.startswith(":quit"):
#             break
#         if raw.startswith(":temp"):
#             try:    temperature = float(raw.split()[1])
#             except: print("Usage: :temp 0.7")
#             print(f"temperature = {temperature}"); continue
#         if raw.startswith(":k"):
#             try:    top_k = int(raw.split()[1])
#             except: print("Usage: :k 40")
#             print(f"top_k = {top_k}"); continue

#         if raw.startswith(":token"):
#             prefix, max_new, mode = raw[6:].strip(), 8, "token"
#         elif raw.startswith(":line"):
#             prefix, max_new, mode = raw[5:].strip(), 64, "line"
#         else:
#             prefix, max_new, mode = raw, 64, "line"

#         inp = hf_tok(prefix, return_tensors="pt").to(device)
#         with torch.no_grad():
#             out = model.generate(
#                 **inp,
#                 max_new_tokens=max_new,
#                 temperature=temperature,
#                 do_sample=(temperature > 0.05),
#                 top_k=top_k,
#                 pad_token_id=hf_tok.eos_token_id,
#             )
#         completion = hf_tok.decode(out[0, inp["input_ids"].size(1):], skip_special_tokens=True)

#         # for :token mode, cut at first whitespace boundary
#         if mode == "token":
#             for i, ch in enumerate(completion):
#                 if ch in " \t\n" and i > 0:
#                     completion = completion[:i]; break
#         # for :line mode, cut at first newline
#         elif mode == "line" and "\n" in completion:
#             completion = completion.split("\n")[0]

#         color = "\033[32m" if mode == "token" else "\033[33m"
#         print(f"  ← {mode:5s} completion: {prefix}{color}{completion}\033[0m\n")

## Main

In [4]:
class Arguments():
    def __init__(self,
                 data_dir:        str   = f"{WORKDIR}/Clean_Dataset",
                 ckpt_dir:        str   = f"{WORKDIR}/checkpoints/{TOKEN_MODEL_NAME}",
                 plot_dir:        str   = f"{WORKDIR}/plots/{TOKEN_MODEL_NAME}",
                 epochs:          int   = 5,
                 batch:           int   = 4,        # CodeGen-350M is large — start small
                 lr:              float = 5e-5,     # LOWER lr for fine-tuning a pretrained model
                 ctx:             int   = 256,      # CodeGen handles long context well
                 max_files:       int   = 0,
                 val_split:       float = 0.1,
                 seed:            int   = 42,
                 label_smoothing: float = 0.1,
                 warmup_frac:     float = 0.05,
                 patience:        int   = 3,
                 use_amp:         bool  = True,
                 skip_token:      bool  = False,
                 test:            bool  = False):
        self.data_dir        = data_dir
        self.ckpt_dir        = ckpt_dir
        self.plot_dir        = plot_dir
        self.epochs          = epochs
        self.batch           = batch
        self.lr              = lr
        self.ctx             = ctx
        self.max_files       = max_files
        self.val_split       = val_split
        self.seed            = seed
        self.label_smoothing = label_smoothing
        self.warmup_frac     = warmup_frac
        self.patience        = patience
        self.use_amp         = use_amp
        self.skip_token      = skip_token
        self.test            = test


def main():
    # args = Arguments()
    args = Arguments(max_files=5, epochs=2)   # quick smoke-test config
    # args = Arguments(test=True)

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)

    # ── HF tokenizer + model ─────────────────────────────────
    print(f"[HF] loading tokenizer + model from {HF_MODEL_NAME} …")
    hf_tok = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
    if hf_tok.pad_token is None:
        hf_tok.pad_token = hf_tok.eos_token   # GPT-style models lack a pad token by default

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tm = AutoModelForCausalLM.from_pretrained(HF_MODEL_NAME).to(device)
        tok_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{TOKEN_MODEL_NAME}_*.pt")))
        if tok_paths:
            ck = torch.load(tok_paths[0], map_location=device, weights_only=False)
            tm.load_state_dict(ck["model_state"])
            print(f"[Loaded] token model from {tok_paths[0]}")
        else:
            print(f"[Test] no fine-tuned ckpt found — using pretrained {HF_MODEL_NAME} as-is")
        hand_test_repl(tm, None, hf_tok, None, device)
        return

    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found."); return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split  = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]

    if not args.skip_token:
        print("  Preparing TOKEN model (CodeGen-350M-mono)")

        # ── datasets ─────────────────────────────────────────
        tr_ds = HFTokenDataset(tr_txt, hf_tok, ctx=args.ctx)
        va_ds = HFTokenDataset(va_txt, hf_tok, ctx=args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,
                           num_workers=0, pin_memory=True, drop_last=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False,
                           num_workers=0, pin_memory=True, drop_last=True)

        # ── model ────────────────────────────────────────────
        tok_model = AutoModelForCausalLM.from_pretrained(HF_MODEL_NAME).to(device)
        tok_model.gradient_checkpointing_enable()   # save VRAM at small speed cost
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.1f}M parameters ({HF_MODEL_NAME})")

        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME, from_hf=True)
        tok_log   = MetricLog()

        print("  Training TOKEN model")
        train_token_model(
            model           = tok_model,
            hf_tokenizer    = hf_tok,
            train_dl        = tr_dl,
            val_dl          = va_dl,
            epochs          = args.epochs,
            lr              = args.lr,
            device          = device,
            saver           = tok_saver,
            log             = tok_log,
            plot_dir        = args.plot_dir,
            label_smoothing = args.label_smoothing,
            warmup_frac     = args.warmup_frac,
            patience        = args.patience,
            use_amp         = args.use_amp,
        )

        # ── interactive test ─────────────────────────────────
        hand_test_repl(tok_model, None, hf_tok, None, device)


main()

[HF] loading tokenizer + model from Salesforce/codegen-350M-mono …
[Loading] Started loading
[Data] loaded 5 files from C:\Users\Roman\Documents\Projects\code_autocomplete/Clean_Dataset
[Loading] Ended loading
  Preparing TOKEN model (CodeGen-350M-mono)


[Tokenising]: 100%|██████████| 4/4 [00:00<00:00, 268.27file/s]


[HFTokenDataset] 4,928 tokens, 4,671 samples


[Tokenising]: 100%|██████████| 1/1 [00:00<00:00, 124.21file/s]

[HFTokenDataset] 3,523 tokens, 3,266 samples


[Token Model] 356.7M parameters (Salesforce/codegen-350M-mono)
  Training TOKEN model
[Token] DataLoader — 1167 train batches, 816 val batches


[Token] Epochs:   0%|          | 0/2 [06:49<?, ?ep/s]


KeyboardInterrupt: 